# 🤖 MedRAX Reproduction & ChestAgentBench Evaluation
### Medical Reasoning Agent for Chest X-ray (ICML 2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Uyhuynh23/MedRAX-ChestAgentBench/blob/main/notebooks/MedRAX_ChestAgentBench_Colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-MedRAX--Reproduction-blue?logo=github)](https://github.com/Uyhuynh23/MedRAX-ChestAgentBench)
[![HuggingFace](https://img.shields.io/badge/Dataset-ChestAgentBench-yellow?logo=huggingface)](https://huggingface.co/datasets/wanglab/chest-agent-bench)
[![arXiv](https://img.shields.io/badge/arXiv-2502.02673-red)](https://arxiv.org/abs/2502.02673)

---

### 📋 Overview & Thesis Objectives
This notebook provides an end-to-end, reproducible pipeline to execute and benchmark **MedRAX** (*Medical Reasoning Agent for Chest X-ray*) on the **ChestAgentBench** dataset directly within **Google Colab**.

Key features of this notebook:
1. **Persistent Google Drive Integration**: Automatically mounts Google Drive and routes heavy model weights (`checkpoints/`), Hugging Face cache (`HF_HOME`), and dataset figures into Drive so you **never have to re-download 30+ GB upon session disconnection**.
2. **Flexible Hardware Profiles**: Supports Colab Free/Pro **T4 / L4 (15–24 GB VRAM)** using 8-bit quantization & selective tools, as well as Colab Pro **A100 (40–80 GB VRAM)** for the full tool suite.
3. **Multi-Model Reproduction**:
   - **Baseline Evaluation**: Multimodal VLM (GPT-4o / Gemini) answering directly from images and clinical history.
   - **MedRAX Agent Evaluation**: Dynamic tool orchestration (classification, segmentation, VQA, report generation) via LangChain/LangGraph.
   - **ACE Playbook Exploration**: Agentic Context Engineering playbook evaluation.
4. **Structured Analytics**: Automatically generates breakdown metrics across 7 clinical competencies (Detection, Classification, Localization, Comparison, Relationship, Diagnosis, Characterization) and saves publication-ready figures to your Drive.



## 1. ⚙️ Hardware & GPU Environment Diagnostics
MedRAX utilizes vision-language models and specialized neural networks. Ensure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU / A100 GPU`).



In [1]:
# Check GPU availability and specifications
import os
import sys
import torch

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU   : {device_name} ({vram_gb:.2f} GB VRAM)")
    if vram_gb < 20:
        print("💡 Recommendation: Using T4/L4 profile with 8-bit quantization and selective tools.")
    else:
        print("🚀 Recommendation: High VRAM detected (A100/L4). Full MedRAX tool suite enabled.")
else:
    print("⚠️ WARNING: No GPU detected! Running on CPU will be extremely slow for vision tools.")

!nvidia-smi



Python Version : 3.13.16
PyTorch Version: 2.14.1+cu130
CUDA Available : True
Detected GPU   : Tesla T4 (14.56 GB VRAM)
💡 Recommendation: Using T4/L4 profile with 8-bit quantization and selective tools.
Fri Nov 13 16:13:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             

## 2. 💾 Google Drive Mounting & Persistent Cache Mapping
Google Colab sessions are ephemeral: when the runtime terminates, all downloaded models and datasets in `/content/` are destroyed.

Here, we map Google Drive to store:
- `checkpoints/`: Heavy model weights (CheXagent, Maira-2, LLaVA-Med, etc.)
- `cache/huggingface/`: Hugging Face transformers cache (`HF_HOME`)
- `data/chestagentbench/`: The 2,500 questions and unzipped `figures/`
- `results/`: Evaluation logs, predictions, accuracy summaries, and plots



In [3]:
# Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# Define persistent storage directories on Google Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/MedRAX_Thesis')

PERSISTENT_DIRS = {
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "hf_cache": DRIVE_ROOT / "cache" / "huggingface",
    "torch_cache": DRIVE_ROOT / "cache" / "torch",
    "data": DRIVE_ROOT / "data" / "chestagentbench",
    "results": DRIVE_ROOT / "results",
    "logs": DRIVE_ROOT / "logs",
}

for name, path in PERSISTENT_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"[Ready] {name:<12} -> {path}")

# Set Global Environment Variables
os.environ["HF_HOME"] = str(PERSISTENT_DIRS["hf_cache"])
os.environ["TRANSFORMERS_CACHE"] = str(PERSISTENT_DIRS["hf_cache"])
os.environ["TORCH_HOME"] = str(PERSISTENT_DIRS["torch_cache"])
os.environ["MEDRAX_MODEL_DIR"] = str(PERSISTENT_DIRS["checkpoints"])
os.environ["BENCHMARK_DATA_DIR"] = str(PERSISTENT_DIRS["data"])
os.environ["MEDRAX_RESULTS_DIR"] = str(PERSISTENT_DIRS["results"])

print("\n✅ Persistent Google Drive environment variables successfully configured!")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Ready] checkpoints  -> /content/drive/MyDrive/MedRAX_Thesis/checkpoints
[Ready] hf_cache     -> /content/drive/MyDrive/MedRAX_Thesis/cache/huggingface
[Ready] torch_cache  -> /content/drive/MyDrive/MedRAX_Thesis/cache/torch
[Ready] data         -> /content/drive/MyDrive/MedRAX_Thesis/data/chestagentbench
[Ready] results      -> /content/drive/MyDrive/MedRAX_Thesis/results
[Ready] logs         -> /content/drive/MyDrive/MedRAX_Thesis/logs

✅ Persistent Google Drive environment variables successfully configured!


## 3. 🐙 Clone Reproduction Repository from GitHub
Next, we clone our Git repository into Colab and navigate to the project root.



In [4]:
# Configure GitHub repository
import os
from pathlib import Path

# Enter your repository URL (or use the reproduction repo)
REPO_URL = "https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git"
REPO_DIR = Path("/content/MedRAX-ChestAgentBench")

if not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} into {REPO_DIR} ...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}. Pulling latest changes...")
    %cd {REPO_DIR}
    !git pull origin main

%cd {REPO_DIR}
!git status



Cloning https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git into /content/MedRAX-ChestAgentBench ...
Cloning into '/content/MedRAX-ChestAgentBench'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 125 (delta 3), reused 125 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 1.03 MiB | 3.41 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/MedRAX-ChestAgentBench
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


### Link Local Repository Folders to Google Drive
We create symbolic links from the repository root to our Google Drive directories. This allows MedRAX scripts to use standard paths while reading and writing directly to Google Drive.



In [5]:
# Run setup_drive script to establish symlinks
!python scripts/setup_drive.py --drive-folder "MedRAX_Thesis"



[Drive Setup] Running in Google Colab environment.
[Drive Setup] Google Drive is already mounted.

 Initializing Persistent Google Drive Architecture
 Root: /content/drive/MyDrive/MedRAX_Thesis
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/checkpoints
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/cache/huggingface
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/cache/torch
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/data/chestagentbench
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/results
  [OK] Directory ready: /content/drive/MyDrive/MedRAX_Thesis/logs

Setting up local workspace symlinks:
  [Symlinked] /content/MedRAX-ChestAgentBench/checkpoints -> /content/drive/MyDrive/MedRAX_Thesis/checkpoints
  [Symlinked] /content/MedRAX-ChestAgentBench/cache -> /content/drive/MyDrive/MedRAX_Thesis/cache
  [Symlinked] /content/MedRAX-ChestAgentBench/data/chestagentbench -> /content/drive/MyDrive/MedRAX_Thesis/data/chesta

## 4. 📦 Install Dependencies & Specialized Radiology Packages
MedRAX requires:
- `transformers` pinned to a specific commit for custom vision-language architectures
- `torchxrayvision` for chest X-ray disease classification (DenseNet-121)
- `langchain`, `langgraph` for agent orchestration
- `accelerate`, `bitsandbytes` for 8-bit memory optimization



In [7]:
%cd /content/MedRAX-ChestAgentBench

# 1. Pull the new 20MB fast-install fix from GitHub
!git pull origin main

# 2. Ensure numpy < 2
!pip install "numpy<2.0.0"

# 3. Fast install using prebuilt wheels and direct zip archive
!pip install -r requirements.txt
!pip install -e . --no-deps

# 4. Verify imports
import medrax
import torchxrayvision as xrv
import langchain
import langgraph
import datasets
print("✅ Core dependencies imported successfully!")

/content/MedRAX-ChestAgentBench
  Using cached numpy-1.26.4.tar.gz (15.8 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... canceled
ERROR: Operation cancelled by user
  Cloning https://github.com/huggingface/transformers.git (to revision 88d960937c81a32bfb63356a2e8ecf7999619681) to /tmp/pip-install-x7f3o4mn/transformers_4e954d8eda7f45e3ba72d29657ade77e
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-install-x7f3o4mn/transformers_4e954d8eda7f45e3ba72d29657ade77e
ERROR: Operation cancelled by user


KeyboardInterrupt: 

## 5. 🔑 Configure API Keys & Secrets
MedRAX uses an LLM (such as GPT-4o or Gemini) as the reasoning orchestrator.
You can configure your keys via Google Colab Secrets (the 🔑 icon on the left panel) or enter them securely below.



In [ ]:
import os
from getpass import getpass

# 1. OpenAI API Key (Required for GPT-4o backbone)
try:
    from google.colab import userdata
    openai_key = userdata.get('OPENAI_API_KEY')
except Exception:
    openai_key = None

if not openai_key:
    openai_key = os.getenv("OPENAI_API_KEY") or getpass("Enter OPENAI_API_KEY (leave blank to skip): ")

if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key
    print("✅ OPENAI_API_KEY configured.")

# 2. Hugging Face Access Token (Required for Maira-2 / gated medical models)
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = None

if not hf_token:
    hf_token = os.getenv("HF_TOKEN") or getpass("Enter HF_TOKEN (leave blank to skip): ")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("✅ HF_TOKEN configured.")

# 3. Google Gemini API Key (Optional alternative backbone)
try:
    google_key = userdata.get('GOOGLE_API_KEY')
except Exception:
    google_key = None

if google_key:
    os.environ["GOOGLE_API_KEY"] = google_key
    print("✅ GOOGLE_API_KEY configured.")



## 6. 📥 Download ChestAgentBench Dataset & Extract Figures
The **ChestAgentBench** benchmark (`wanglab/chest-agent-bench`) contains:
- `metadata.jsonl`: 2,500 multiple-choice questions across 609 Eurorad clinical cases.
- `figures.zip`: High-resolution thoracic radiographs and CT/imaging slices.

The script below checks your Google Drive first so it **only downloads once**.



In [ ]:
# Download dataset into persistent Google Drive storage
!python scripts/download_benchmark.py --data-dir "$BENCHMARK_DATA_DIR"



## 7. 🔍 Interactive Dataset Exploration & Inspection
Let's inspect a few sample cases, questions, choices, and their corresponding radiographs.



In [ ]:
import json
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

data_dir = Path(os.environ["BENCHMARK_DATA_DIR"])
metadata_path = data_dir / "metadata.jsonl"
figures_dir = data_dir / "figures"

# Read sample questions
questions = []
with open(metadata_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            questions.append(json.loads(line))

print(f"Total questions loaded: {len(questions)}")

# Select a random question to visualize
sample = random.choice(questions)

print(f"\n--- Sample Clinical Case: {sample.get('case_id')} | Question ID: {sample.get('question_id')} ---")
print(f"Categories: {sample.get('categories')}")
print(f"Question:\n{sample.get('question')}")
print(f"\nGround Truth Answer: {sample.get('answer')}")
print(f"Clinical Explanation:\n{sample.get('explanation')[:300]}...")

# Display images associated with this question
images = sample.get("images", [])
if isinstance(images, str):
    images = [images]

if images:
    fig, axes = plt.subplots(1, len(images), figsize=(5 * len(images), 5))
    if len(images) == 1:
        axes = [axes]
    for idx, img_rel in enumerate(images):
        clean_name = img_rel.replace("figures/", "") if img_rel.startswith("figures/") else img_rel
        img_path = figures_dir / clean_name
        if img_path.exists():
            img = Image.open(img_path)
            axes[idx].imshow(img, cmap="gray")
            axes[idx].set_title(f"Figure: {clean_name}")
            axes[idx].axis("off")
        else:
            axes[idx].text(0.5, 0.5, "Image not found", ha="center")
            axes[idx].axis("off")
    plt.tight_layout()
    plt.show()



## 8. 🧪 Benchmark Execution & Model Evaluation

We provide two primary evaluation workflows:
- **Option A: Quickstart Baseline (Direct Multimodal LLM)**: Evaluates GPT-4o directly on visual questions without external tool calling.
- **Option B: Full MedRAX Agent**: Evaluates the MedRAX agent with dynamic chest radiology tool calling (classification, segmentation, report generation, VQA).

*Tip: For an initial test, set `--max-cases 2` to verify everything end-to-end in ~1 minute, then remove the flag for full evaluation.*



In [ ]:
# -------------------------------------------------------------
# Option A: Run Baseline MLLM Evaluation (Smoke Test: 2 Cases)
# -------------------------------------------------------------
!python scripts/run_evaluation.py \
    --mode baseline \
    --model chatgpt-4o-latest \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/baseline_test" \
    --max-cases 2



In [ ]:
# -------------------------------------------------------------
# Option B: Run MedRAX Agent with Specialized Radiology Tools
# (Automatically uses model checkpoints stored on Google Drive)
# -------------------------------------------------------------
!python scripts/run_evaluation.py \
    --mode medrax \
    --model chatgpt-4o-latest \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/medrax_test" \
    --max-cases 2



## 9. 📊 Results Analytics & Visualizing Accuracy by Clinical Axes
MedRAX evaluates clinical reasoning across 7 core axes:
1. **Detection**
2. **Classification**
3. **Localization**
4. **Comparison**
5. **Relationship**
6. **Diagnosis**
7. **Characterization**

Let's compute and plot performance across all categories and compare against baseline runs.



In [ ]:
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path(os.environ["MEDRAX_RESULTS_DIR"])

def display_run_summary(run_folder_name):
    summary_path = results_dir / run_folder_name / "summary.json"
    if not summary_path.exists():
        print(f"Summary file not found at: {summary_path}")
        return

    with open(summary_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"============================================================")
    print(f" Run: {run_folder_name} | Mode: {data.get('mode')} | Model: {data.get('model')}")
    print(f" Overall Accuracy: {data.get('overall_accuracy')}% ({data.get('total_correct')}/{data.get('total_evaluated')})")
    print(f"============================================================")

    cat_data = []
    for cat, info in data.get("category_metrics", {}).items():
        cat_data.append({
            "Category": cat,
            "Accuracy (%)": info.get("accuracy", 0.0),
            "Correct": info.get("correct", 0),
            "Total": info.get("total", 0)
        })

    df = pd.DataFrame(cat_data).sort_values("Accuracy (%)", ascending=False)
    display(df)

    # Plot accuracy chart
    plt.figure(figsize=(10, 4))
    sns.set_theme(style="whitegrid")
    bar = sns.barplot(data=df, x="Category", y="Accuracy (%)", palette="viridis")
    plt.title(f"Accuracy Across Clinical Competency Axes ({run_folder_name})", fontsize=14, pad=12)
    plt.ylim(0, 100)
    plt.xticks(rotation=25)
    for p in bar.patches:
        bar.annotate(f"{p.get_height():.1f}%",
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10, xytext=(0, 3),
                     textcoords='offset points')
    plt.tight_layout()
    chart_path = results_dir / run_folder_name / "accuracy_breakdown.png"
    plt.savefig(chart_path, dpi=300)
    print(f"Saved plot to: {chart_path}")
    plt.show()

# Visualize the latest test run
display_run_summary("baseline_test")



## 10. 🔄 Synchronize Results & Updates to GitHub
To preserve new experiment configurations, custom prompt modifications, or evaluation playbooks in your GitHub repository, commit your changes using the cell below.



In [ ]:
# Configure Git credentials in Colab
!git config --global user.name "Uyhuynh23"
!git config --global user.email "htuy23@apcs.fitus.edu.vn"

# Check what has changed in the workspace
!git status

# To push changes:
# 1. Create a GitHub Personal Access Token (PAT) with repo scope at https://github.com/settings/tokens
# 2. Uncomment and run the lines below:
# !git add configs/ scripts/ notebooks/
# !git commit -m "Add reproduction evaluation experiments and notebook updates"
# !git push https://<YOUR_GITHUB_TOKEN>@github.com/Uyhuynh23/MedRAX-ChestAgentBench.git main

